## 🧠 Graph Construction for Multilayer GNN Classification (Naples Cohort)

This section prepares graph inputs for classifying MS vs Control subjects using GNNs based on nodal metrics extracted from two-layer brain networks (GM + rsfMRI).  
It generates two graph types per subject:

- **MULTILAYER**: fully connected graphs with 152 nodes (76 per layer), combining intra- and inter-layer edges.
- **MULTINEXT**: graphs with only diagonal inter-layer links (i.e., node_i ↔ node_i+76).

### 📌 Steps:

1. **Load nodal metrics and clinical labels**:
   - Loads `.csv` files containing nodal metrics per subject and clinical group labels (`GROUP`: 0 = Control, 1 = MS).

2. **Define graph structure**:
   - `edge_index_full`: Fully connected undirected graph across 152 nodes (MULTILAYER).
   - `edge_index_inter`: Only inter-layer diagonal connections (node_i ↔ node_i+76), used for MULTINEXT.

3. **Create PyTorch Geometric Data objects**:
   - For each subject and each mode (`MULTILAYER`, `MULTINEXT`):
     - Extracts nodal features (`degree`, `strength`, `betweenness`, `closeness`, `eigenvector`).
     - Constructs the graph with appropriate `edge_index` and assigns label.
     - Stores graph in the corresponding `data_list`.

4. **Output**:
   - `data_list_multilayer`: List of graphs for fully connected multilayer models.
   - `data_list_multinetx`: List of graphs for inter-layer-only connectivity models.

✅ Ready for training GNNs on MULTILAYER and MULTINEXT architectures.


In [1]:
#  Required libraries
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data

# ----------------------------------------
# 1. Load nodal metrics and clinical labels
# ----------------------------------------

metrics_path = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/nodal_metrics_with_subjects.csv"  
clinical_path = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/CLINIC_Naples_B.csv" 

nodal_df = pd.read_csv(metrics_path)
clinical_df = pd.read_csv(clinical_path)
clinical_df["ID"] = clinical_df["ID"].astype(str)
id_to_label = dict(zip(clinical_df["ID"], clinical_df["GROUP"]))

# ----------------------------------------
# 2. Define graph structure
# ----------------------------------------

N = 76  # number of brain regions per layer
metric_cols = ["degree", "strength", "betweenness", "closeness", "eigenvector"]

# Fully connected edge_index for 152 nodes (intra + inter)
node_ids = torch.arange(152)
edge_index_full = torch.combinations(node_ids, r=2).T
edge_index_full = torch.cat([edge_index_full, edge_index_full[[1, 0], :]], dim=1)  # bidirectional

# Inter-layer index (diagonal only: node_i <-> node_i+76)
inter_layer_pairs = [(i, i+76) for i in range(76)]
edge_index_inter = torch.tensor(inter_layer_pairs, dtype=torch.long).T
edge_index_inter = torch.cat([edge_index_inter, edge_index_inter[[1, 0], :]], dim=1)

# ----------------------------------------
# 3. Create data_list for each type
# ----------------------------------------

data_list_multilayer = []
data_list_multinetx = []

for subject in nodal_df["Subject"].unique():
    for mode in ["MULTILAYER", "MULTINEXT"]:
        df_sub = nodal_df[(nodal_df["Subject"] == subject) & (nodal_df["Layer"] == mode)]
        if len(df_sub) != 152 or subject not in id_to_label:
            continue  # skip incomplete or missing labels

        df_sub = df_sub.sort_values("Node")
        x = torch.tensor(df_sub[metric_cols].values, dtype=torch.float32)
        y = torch.tensor([id_to_label[subject]], dtype=torch.long)

        if mode == "MULTILAYER":
            data = Data(x=x, edge_index=edge_index_full.clone(), y=y)
            data.subject_id = subject 
            data_list_multilayer.append(data)
        elif mode == "MULTINEXT":
            data = Data(x=x, edge_index=edge_index_inter.clone(), y=y)
            data.subject_id = subject 
            data_list_multinetx.append(data)

print(f"✅ Created {len(data_list_multilayer)} MULTILAYER graphs.")
print(f"✅ Created {len(data_list_multinetx)} MULTINEXT graphs.")



✅ Created 105 MULTILAYER graphs.
✅ Created 105 MULTINEXT graphs.


In [2]:
print(nodal_df["Layer"].unique())

['MULTILAYER' 'MULTINEXT']


## 🧠 2D Visualization of Nodal Differences – MULTILAYER vs MULTINEXT (Naples)

This script generates 2D axial brain plots of group-level differences (Control − MS) in nodal graph metrics for multilayer networks.  
It compares two fusion strategies:
- **MULTILAYER**: fully connected supra-graph across both layers.
- **MULTINEXT**: diagonal-only inter-layer connections.

### 🔍 Main Steps:

1. **Load MNI coordinates**:
   - Reads 3D coordinates (X, Y, Z) for 76 brain regions from a `.node` file.
   - Used to locate brain nodes spatially in the plots.

2. **Prepare output directory**:
   - Defines the folder where all `.png` figures will be saved.

3. **Generate per-layer visualizations**:
   - For each **graph metric** (`Degree`, `Strength`, etc.) and **layer** (`GM`, `rsfMRI`):
     - Averages nodal values across Control and MS subjects.
     - Computes group difference:  
       \[
       \Delta = \text{Control Mean} - \text{MS Mean}
       \]
     - Uses `nilearn.plot_connectome()` to display **node-only plots** with no edges:
       - 🔴 Red = Higher in Control
       - 🔵 Blue = Higher in MS

4. **Save outputs**:
   - Each plot is saved as `.png`, labeled by mode, metric, and layer.




In [3]:
#  Required libraries
from nilearn import plotting
import matplotlib.pyplot as plt
import os

# ----------------------------------------
# 1. Load MNI coordinates if not already loaded
# ----------------------------------------

if 'mni_coords' not in globals():
    mni_coords_path = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/Node_mindboggle_default.node"
    with open(mni_coords_path, "r") as f:
        lines = f.readlines()
    mni_coords = []
    for line in lines:
        parts = line.strip().split('\t')
        try:
            coord = list(map(float, parts[:3]))
            mni_coords.append(coord)
        except ValueError:
            continue
    mni_coords = np.array(mni_coords)

# ----------------------------------------
# 2. Define output directory
# ----------------------------------------

output_dir = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML"
os.makedirs(output_dir, exist_ok=True)

# ----------------------------------------
# 3. Visualize for MULTILAYER and MULTINEXT
# ----------------------------------------

metric_names = ["Degree", "Strength", "Betweenness", "Closeness", "Eigenvector"]
layer_names = ["GM", "rsfMRI"]

for mode, data_list in [("MULTILAYER", data_list_multilayer), ("MULTINEXT", data_list_multinetx)]:
    print(f"\n🧠 Generating plots for: {mode}")

    control_graphs = [data for data in data_list if data.y.item() == 0]
    ms_graphs = [data for data in data_list if data.y.item() == 1]

    control_features = torch.stack([g.x for g in control_graphs])
    ms_features = torch.stack([g.x for g in ms_graphs])

    for metric_index, metric_name in enumerate(metric_names):
        for layer_idx, layer_label in enumerate(layer_names):
            start = layer_idx * 76
            end = start + 76

            mean_control = control_features[:, start:end, metric_index].mean(dim=0).numpy()
            mean_ms = ms_features[:, start:end, metric_index].mean(dim=0).numpy()
            delta = mean_control - mean_ms

            empty_adj = np.zeros((76, 76))
            title = f"{mode} - Δ {metric_name} (Control − MS, capa {layer_label})\nRed: ↑ Control | Blue: ↑ MS"

            display = plotting.plot_connectome(empty_adj, mni_coords,
                                               node_color=delta,
                                               node_size=40,
                                               edge_threshold=None,
                                               title=title)

            filename = f"{mode}_delta_{metric_name.lower()}_{layer_label.lower()}_control_minus_ms.png"
            output_file = os.path.join(output_dir, filename)
            plt.savefig(output_file, dpi=300)
            plt.close()
            print(f"✅ Saved: {output_file}")




🧠 Generating plots for: MULTILAYER
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML\MULTILAYER_delta_degree_gm_control_minus_ms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML\MULTILAYER_delta_degree_rsfmri_control_minus_ms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML\MULTILAYER_delta_strength_gm_control_minus_ms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML\MULTILAYER_delta_strength_rsfmri_control_minus_ms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summar

## 🧠 Compute and Save Nodal Group Differences — MULTILAYER vs MULTINEXT (Naples)

This script computes group-level nodal differences (Control − MS) across brain graph metrics for two multilayer network models:  
- **MULTILAYER** (fully connected supra-graph)  
- **MULTINEXT** (inter-layer diagonal links only)

### 🔍 Main Steps:

1. **Set output path**:
   - All results will be saved in a common `figures/ML` directory as `.csv` tables.

2. **Define metrics and layers**:
   - Graph metrics analyzed: `Degree`, `Strength`, `Betweenness`, `Closeness`, `Eigenvector`.
   - Sublayers: `GM` (Morphological) and `rsfMRI` (Functional).

3. **Loop through each model (MULTILAYER, MULTINEXT)**:
   - Filter graphs by clinical group (Control vs MS).
   - For each graph metric and sublayer:
     - Compute mean nodal values for both groups.
     - Calculate group difference:  
       \[
       \Delta = \text{Mean}_{\text{Control}} - \text{Mean}_{\text{MS}}
       \]
     - Store node-level Δ values with corresponding metadata.

4. **Export results**:
   - Output is saved as a `.csv` file named:  
     `MULTILAYER_nodal_deltas_control_minus_ms.csv` or  
     `MULTINEXT_nodal_deltas_control_minus_ms.csv`.



In [4]:

# Define output directory
output_dir = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML"
os.makedirs(output_dir, exist_ok=True)

# Define metric names
metric_names = ["Degree", "Strength", "Betweenness", "Closeness", "Eigenvector"]
layer_names = ["GM", "rsfMRI"]

# Define data sources
multi_data = {
    "MULTILAYER": data_list_multilayer,
    "MULTINEXT": data_list_multinetx
}

# Loop over each multilayer type
for mode, data_list in multi_data.items():
    if data_list is None or len(data_list) == 0:
        print(f"⚠️ Skipping {mode}: no data available.")
        continue

    print(f"\n📊 Computing nodal deltas for: {mode}")
    control_graphs = [g for g in data_list if g.y.item() == 0]
    ms_graphs = [g for g in data_list if g.y.item() == 1]

    control_features = torch.stack([g.x for g in control_graphs])
    ms_features = torch.stack([g.x for g in ms_graphs])

    all_deltas = []

    for metric_index, metric_name in enumerate(metric_names):
        for layer_idx, sublayer in enumerate(layer_names):
            start = layer_idx * 76
            end = start + 76

            mean_control = control_features[:, start:end, metric_index].mean(dim=0).numpy()
            mean_ms = ms_features[:, start:end, metric_index].mean(dim=0).numpy()
            delta = mean_control - mean_ms

            for node in range(76):
                all_deltas.append({
                    "Network": mode,
                    "Sublayer": sublayer,
                    "Node": node,
                    "Metric": metric_name,
                    "Mean_Control": mean_control[node],
                    "Mean_MS": mean_ms[node],
                    "Delta_Control_minus_MS": delta[node]
                })

    # Save to CSV
    df = pd.DataFrame(all_deltas)
    filename = f"{mode}_nodal_deltas_control_minus_ms.csv"
    csv_path = os.path.join(output_dir, filename)
    df.to_csv(csv_path, index=False)
    print(f"✅ CSV saved: {csv_path}")



📊 Computing nodal deltas for: MULTILAYER
✅ CSV saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML\MULTILAYER_nodal_deltas_control_minus_ms.csv

📊 Computing nodal deltas for: MULTINEXT
✅ CSV saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML\MULTINEXT_nodal_deltas_control_minus_ms.csv


## 🧠 2D Nodal Visualization — RRMS vs SPMS (Multilayer Graphs, Naples)

This block generates axial 2D plots of nodal differences between **RRMS** and **SPMS** patients for multilayer brain networks using two models:  
- `MULTILAYER` (fully connected supra-graph)  
- `MULTINEXT` (inter-layer links only)

### 🔍 Main Steps:

1. **Output directory**:
   - Ensures the destination folder exists to store the generated images.

2. **Select subjects by MS subtype**:
   - Extracts IDs of subjects labeled as `RRMS` and `SPMS` from the `nodal_df` table.

3. **Loop over fusion models and sublayers**:
   - For each fusion model (`MULTILAYER`, `MULTINEXT`) and each sublayer (`GM`, `rsfMRI`):
     - Aggregates graph metric values (e.g., Degree, Strength) for RRMS and SPMS groups.
     - Computes the difference:  
       \[
       \Delta = \text{Mean}_{\text{RRMS}} - \text{Mean}_{\text{SPMS}}
       \]
     - Plots only the nodes (no edges) using MNI coordinates and `nilearn`.

4. **Color encoding**:
   - 🔴 Red = metric higher in **RRMS**  
   - 🔵 Blue = metric higher in **SPMS**

5. **Save outputs**:
   - Each plot is saved as `.png`, with filenames indicating the model, layer, metric, and group comparison.

✅ This visualization highlights regional brain alterations between MS subtypes in a spatially interpretable format.


In [5]:

# ----------------------------------------
# 1. Define output directory
# ----------------------------------------

output_dir = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML"
os.makedirs(output_dir, exist_ok=True)

# ----------------------------------------
# 2. Get RRMS and SPMS subject IDs
# ----------------------------------------

subject_labels = nodal_df[["Subject", "mstype_label"]].drop_duplicates()
rrms_subjects = subject_labels[subject_labels["mstype_label"] == "RRMS"]["Subject"].tolist()
spms_subjects = subject_labels[subject_labels["mstype_label"] == "SPMS"]["Subject"].tolist()

# ----------------------------------------
# 3. Define metrics and sublayers
# ----------------------------------------

metric_names = ["Degree", "Strength", "Betweenness", "Closeness", "Eigenvector"]
layer_names = ["GM", "rsfMRI"]

# ----------------------------------------
# 4. Loop over MULTILAYER and MULTINEXT
# ----------------------------------------

multi_data = {
    "MULTILAYER": data_list_multilayer,
    "MULTINEXT": data_list_multinetx
}

for mode, data_list in multi_data.items():
    print(f"\n🧠 Generating RRMS − SPMS plots for: {mode}")
    
    rrms_graphs = [g for g in data_list if g.subject_id in rrms_subjects]
    spms_graphs = [g for g in data_list if g.subject_id in spms_subjects]

    if not rrms_graphs or not spms_graphs:
        print(f"⚠️ Skipping {mode} (missing RRMS or SPMS subjects)")
        continue

    rrms_features = torch.stack([g.x for g in rrms_graphs])
    spms_features = torch.stack([g.x for g in spms_graphs])

    for metric_index, metric_name in enumerate(metric_names):
        for layer_idx, layer_label in enumerate(layer_names):
            start = layer_idx * 76
            end = start + 76

            mean_rrms = rrms_features[:, start:end, metric_index].mean(dim=0).numpy()
            mean_spms = spms_features[:, start:end, metric_index].mean(dim=0).numpy()
            delta = mean_rrms - mean_spms

            empty_adj = np.zeros((76, 76))
            title = f"{mode} - Δ {metric_name} (RRMS − SPMS, capa {layer_label})\nRed: ↑ RRMS | Blue: ↑ SPMS"

            display = plotting.plot_connectome(empty_adj, mni_coords,
                                               node_color=delta,
                                               node_size=40,
                                               edge_threshold=None,
                                               title=title)

            filename = f"{mode}_delta_{metric_name.lower()}_{layer_label.lower()}_rrms_minus_spms.png"
            filepath = os.path.join(output_dir, filename)
            plt.savefig(filepath, dpi=300)
            plt.close()
            print(f"✅ Saved: {filepath}")



🧠 Generating RRMS − SPMS plots for: MULTILAYER
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML\MULTILAYER_delta_degree_gm_rrms_minus_spms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML\MULTILAYER_delta_degree_rsfmri_rrms_minus_spms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML\MULTILAYER_delta_strength_gm_rrms_minus_spms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML\MULTILAYER_delta_strength_rsfmri_rrms_minus_spms.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/noda

## 🧠 2D Visualization — EDSS Mild vs Severe (Multilayer Graphs, Naples)

This block generates axial 2D plots of nodal graph metric differences between **Mild** and **Severe** EDSS groups for each **multilayer brain network type** (`MULTILAYER`, `MULTINEXT`) and each **sublayer** (`GM`, `rsfMRI`).

### 🔍 Main Steps:

1. **Output directory**:
   - Ensures the output folder exists to save visualizations.

2. **Subject filtering**:
   - Extracts subjects labeled as `0–2 (Mild)` and `6.5–9 (Severe)` from the `edss_group` column.

3. **Metric computation per sublayer**:
   - For each graph metric (`Degree`, `Strength`, `Betweenness`, etc.) and sublayer:
     - Computes the mean nodal value across subjects in each group.
     - Calculates group difference:  
       \[
       \Delta = \text{Mean}_{\text{Mild}} - \text{Mean}_{\text{Severe}}
       \]

4. **2D brain plotting**:
   - Uses `nilearn.plot_connectome()` with an empty adjacency matrix.
   - Colors represent nodal Δ values:
     - 🔴 Red = higher in Mild
     - 🔵 Blue = higher in Severe
   - Title includes the metric, layer, and EDSS group comparison.

5. **Output**:
   - Saves each figure as a `.png` image.
   - Filenames include network type, metric, sublayer, and comparison label.

✅ These plots allow spatial inspection of regional nodal differences in patients with different disability severity levels.


In [6]:


# ----------------------------------------
# 1. Output directory
# ----------------------------------------

output_dir = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML"
os.makedirs(output_dir, exist_ok=True)

# ----------------------------------------
# 2. Filter subjects with EDSS labels
# ----------------------------------------

subject_labels = nodal_df[["Subject", "edss_group"]].drop_duplicates()
mild_subjects = subject_labels[subject_labels["edss_group"] == "0–2 (Mild)"]["Subject"].tolist()
severe_subjects = subject_labels[subject_labels["edss_group"] == "6.5–9 (Severe)"]["Subject"].tolist()

# ----------------------------------------
# 3. Metric and layer names
# ----------------------------------------

metric_names = ["Degree", "Strength", "Betweenness", "Closeness", "Eigenvector"]
layer_names = ["GM", "rsfMRI"]

# ----------------------------------------
# 4. Loop over multilayer networks
# ----------------------------------------

multi_data = {
    "MULTILAYER": data_list_multilayer,
    "MULTINEXT": data_list_multinetx
}

for mode, data_list in multi_data.items():
    print(f"\n🧠 Generating EDSS Mild − Severe plots for: {mode}")

    mild_graphs = [g for g in data_list if g.subject_id in mild_subjects]
    severe_graphs = [g for g in data_list if g.subject_id in severe_subjects]

    if not mild_graphs or not severe_graphs:
        print(f"⚠️ Skipping {mode} (missing Mild or Severe subjects)")
        continue

    mild_features = torch.stack([g.x for g in mild_graphs])
    severe_features = torch.stack([g.x for g in severe_graphs])

    for i, metric in enumerate(metric_names):
        for layer_idx, layer_label in enumerate(layer_names):
            start = layer_idx * 76
            end = start + 76

            mean_mild = mild_features[:, start:end, i].mean(dim=0).numpy()
            mean_severe = severe_features[:, start:end, i].mean(dim=0).numpy()
            delta = mean_mild - mean_severe

            title = f"{mode} - Δ {metric} (EDSS Mild − Severe, capa {layer_label})\nRed = ↑ Mild | Blue = ↑ Severe"
            empty_adj = np.zeros((76, 76))

            display = plotting.plot_connectome(empty_adj, mni_coords,
                                               node_color=delta,
                                               node_size=40,
                                               edge_threshold=None,
                                               title=title)

            filename = f"{mode}_delta_{metric.lower()}_{layer_label.lower()}_edss_mild_minus_severe.png"
            filepath = os.path.join(output_dir, filename)
            plt.savefig(filepath, dpi=300)
            plt.close()
            print(f"✅ Saved: {filepath}")



🧠 Generating EDSS Mild − Severe plots for: MULTILAYER
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML\MULTILAYER_delta_degree_gm_edss_mild_minus_severe.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML\MULTILAYER_delta_degree_rsfmri_edss_mild_minus_severe.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML\MULTILAYER_delta_strength_gm_edss_mild_minus_severe.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML\MULTILAYER_delta_strength_rsfmri_edss_mild_minus_severe.png
✅ Saved: F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILA

## 🧠 GNN Classification – Multilayer Graphs (Naples Cohort)

This script implements training and evaluation of **three Graph Neural Network (GNN) architectures** on **multilayer brain networks** (`MULTILAYER`, `MULTINEXT`), using nodal graph metrics as input features.

---

### 🔧 Required Libraries

- PyTorch & PyTorch Geometric: GCNConv, SAGEConv, GATConv, DataLoader
- Scikit-learn: StratifiedKFold, classification metrics (Accuracy, F1, AUC, etc.)

---

### 🛠️ Main Steps

#### 1. **Set Random Seeds**
Ensures reproducibility by fixing seeds for NumPy, Python, and PyTorch.

#### 2. **Define GNN Architectures**

- `GCN`: 2 convolution layers using `GCNConv`, with dropout and global mean pooling.
- `GraphSAGE`: uses `SAGEConv` layers for neighborhood aggregation.
- `GAT`: attention-based model with `GATConv` layers and multi-head mechanism.

Each architecture outputs a 2-class softmax prediction (Control vs MS).

#### 3. **Training & Testing Functions**

- `train(model, loader, optimizer, device)`: trains one epoch using cross-entropy loss.
- `test(model, loader, device)`: evaluates model and collects predictions, probabilities, and labels.

#### 4. **Stratified Cross-Validation (run_cross_validation)**

- Applies **Stratified K-Fold** (k = 5) to preserve class distribution in each fold.
- For each fold:
  - Trains model (`epochs = 100`)
  - Evaluates on held-out test set
  - Computes performance metrics:  
    `Accuracy`, `F1`, `Precision`, `Recall`, `AUC`
- Saves per-fold metrics and average summary to `.csv` file.

#### 5. **Execution for All Networks & Models**

- Trains `GCN`, `GraphSAGE`, and `GAT` models on both:
  - `MULTILAYER`: fully connected graphs (intra + inter)
  - `MULTINEXT`: diagonal-only interlayer edges
- Results are stored in:



In [7]:
# Required libraries
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, global_mean_pool
from torch_geometric.loader import DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, precision_score, recall_score
import random


# ----------------------------------------
# 1. Set random seeds
# ----------------------------------------

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# ----------------------------------------
# 2. Define GNN architectures
# ----------------------------------------

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, 2)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        x = global_mean_pool(x, batch)
        return self.lin(x)

class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.lin = torch.nn.Linear(hidden_channels, 2)

    def forward(self, x, edge_index, batch):
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        x = global_mean_pool(x, batch)
        return self.lin(x)

class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(GAT, self).__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=4, concat=True)
        self.conv2 = GATConv(hidden_channels * 4, hidden_channels, heads=1)
        self.lin = torch.nn.Linear(hidden_channels, 2)

    def forward(self, x, edge_index, batch):
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        x = global_mean_pool(x, batch)
        return self.lin(x)

# ----------------------------------------
# 3. Training and evaluation
# ----------------------------------------

def train(model, loader, optimizer, device):
    model.train()
    for data in loader:
        data = data.to(device)
        optimizer.zero_grad()
        out = model(data.x, data.edge_index, data.batch)
        loss = F.cross_entropy(out, data.y)
        loss.backward()
        optimizer.step()

def test(model, loader, device):
    model.eval()
    y_true, y_pred, y_prob = [], [], []
    with torch.no_grad():
        for data in loader:
            data = data.to(device)
            out = model(data.x, data.edge_index, data.batch)
            prob = F.softmax(out, dim=1)
            pred = prob.argmax(dim=1)
            y_true.extend(data.y.cpu().numpy())
            y_pred.extend(pred.cpu().numpy())
            y_prob.extend(prob[:, 1].cpu().numpy())
    return y_true, y_pred, y_prob

# ----------------------------------------
# 4. Cross-validation loop
# ----------------------------------------

def run_cross_validation(ModelClass, model_name, data_list, layer, output_dir, k=5, hidden_dim=32, epochs=100):
    print(f"\n🔁 Running {model_name} for {layer} with {k}-Fold Stratified CV")
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    labels = [data.y.item() for data in data_list]

    fold_results = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(data_list, labels)):
        train_data = [data_list[i] for i in train_idx]
        test_data = [data_list[i] for i in test_idx]

        train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
        test_loader = DataLoader(test_data, batch_size=16)

        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = ModelClass(in_channels=5, hidden_channels=hidden_dim).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

        for epoch in range(epochs):
            train(model, train_loader, optimizer, device)

        y_true, y_pred, y_prob = test(model, test_loader, device)

        metrics = {
            "Model": model_name,
            "Network": layer,
            "Fold": fold + 1,
            "Accuracy": accuracy_score(y_true, y_pred),
            "F1": f1_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred),
            "Recall": recall_score(y_true, y_pred),
            "AUC": roc_auc_score(y_true, y_prob)
        }
        fold_results.append(metrics)
        print(f"📊 Fold {fold+1} - Acc: {metrics['Accuracy']:.3f}, F1: {metrics['F1']:.3f}, AUC: {metrics['AUC']:.3f}")

    # Save to CSV
    df = pd.DataFrame(fold_results)
    avg = df.iloc[:, 3:].mean().to_dict()
    avg_row = {
        "Model": model_name,
        "Network": layer,
        "Fold": "Average",
        **avg
    }
    df = pd.concat([df, pd.DataFrame([avg_row])], ignore_index=True)

    print(f"\n📋 Results for {model_name} - {layer}:")
    print(df.round(3).to_string(index=False))

    output_path = os.path.join(output_dir, f"{layer}_GNN_metrics_folds.csv")    
    #df.to_csv(output_path, index=False)
    #print(f"✅ Saved to {output_path}")
    # Append to CSV if it exists, otherwise write with header
    df.to_csv(output_path, index=False, mode='a', header=not os.path.exists(output_path))
    print(f"✅ Appended results to {output_path}")
    
# ----------------------------------------
# 5. Run models for multilayer graphs
# ----------------------------------------

output_dir = "F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML"
os.makedirs(output_dir, exist_ok=True)

multilayer_data = {
    "MULTILAYER": data_list_multilayer,
    "MULTINEXT": data_list_multinetx
}

for layer_name, data_list in multilayer_data.items():
    run_cross_validation(GCN, "GCN", data_list, layer_name, output_dir)
    run_cross_validation(GraphSAGE, "GraphSAGE", data_list, layer_name, output_dir)
    run_cross_validation(GAT, "GAT", data_list, layer_name, output_dir)



🔁 Running GCN for MULTILAYER with 5-Fold Stratified CV
📊 Fold 1 - Acc: 0.571, F1: 0.308, AUC: 0.518
📊 Fold 2 - Acc: 0.524, F1: 0.167, AUC: 0.782
📊 Fold 3 - Acc: 0.429, F1: 0.600, AUC: 0.636
📊 Fold 4 - Acc: 0.619, F1: 0.500, AUC: 0.582
📊 Fold 5 - Acc: 0.619, F1: 0.429, AUC: 0.609

📋 Results for GCN - MULTILAYER:
Model    Network    Fold  Accuracy    F1  Precision  Recall   AUC
  GCN MULTILAYER       1     0.571 0.308      1.000   0.182 0.518
  GCN MULTILAYER       2     0.524 0.167      1.000   0.091 0.782
  GCN MULTILAYER       3     0.429 0.600      0.450   0.900 0.636
  GCN MULTILAYER       4     0.619 0.500      0.667   0.400 0.582
  GCN MULTILAYER       5     0.619 0.429      0.750   0.300 0.609
  GCN MULTILAYER Average     0.552 0.401      0.773   0.375 0.625
✅ Appended results to F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML\MULTILAYER_GNN_metrics_folds.csv

🔁 Running GraphSAGE f

f:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


📊 Fold 3 - Acc: 0.524, F1: 0.000, AUC: 0.473
📊 Fold 4 - Acc: 0.476, F1: 0.421, AUC: 0.591
📊 Fold 5 - Acc: 0.524, F1: 0.667, AUC: 0.682

📋 Results for GAT - MULTILAYER:
Model    Network    Fold  Accuracy    F1  Precision  Recall   AUC
  GAT MULTILAYER       1     0.571 0.308      1.000   0.182 0.682
  GAT MULTILAYER       2     0.429 0.000      0.000   0.000 0.700
  GAT MULTILAYER       3     0.524 0.000      0.000   0.000 0.473
  GAT MULTILAYER       4     0.476 0.421      0.444   0.400 0.591
  GAT MULTILAYER       5     0.524 0.667      0.500   1.000 0.682
  GAT MULTILAYER Average     0.505 0.279      0.389   0.316 0.625
✅ Appended results to F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML\MULTILAYER_GNN_metrics_folds.csv

🔁 Running GCN for MULTINEXT with 5-Fold Stratified CV
📊 Fold 1 - Acc: 0.524, F1: 0.688, AUC: 0.436


f:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


📊 Fold 2 - Acc: 0.476, F1: 0.000, AUC: 0.382
📊 Fold 3 - Acc: 0.476, F1: 0.645, AUC: 0.664
📊 Fold 4 - Acc: 0.476, F1: 0.645, AUC: 0.491
📊 Fold 5 - Acc: 0.476, F1: 0.645, AUC: 0.482

📋 Results for GCN - MULTINEXT:
Model   Network    Fold  Accuracy    F1  Precision  Recall   AUC
  GCN MULTINEXT       1     0.524 0.688      0.524     1.0 0.436
  GCN MULTINEXT       2     0.476 0.000      0.000     0.0 0.382
  GCN MULTINEXT       3     0.476 0.645      0.476     1.0 0.664
  GCN MULTINEXT       4     0.476 0.645      0.476     1.0 0.491
  GCN MULTINEXT       5     0.476 0.645      0.476     1.0 0.482
  GCN MULTINEXT Average     0.486 0.525      0.390     0.8 0.491
✅ Appended results to F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML\MULTINEXT_GNN_metrics_folds.csv

🔁 Running GraphSAGE for MULTINEXT with 5-Fold Stratified CV


...

📊 Fold 1 - Acc: 0.476, F1: 0.000, AUC: 0.436


f:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


📊 Fold 2 - Acc: 0.476, F1: 0.000, AUC: 0.391
📊 Fold 3 - Acc: 0.476, F1: 0.645, AUC: 0.355


f:\Cursos\UOC Master Bio Inf. Est\M0.209 - UOC - TFM - Bio ML - Grafos\venv\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


📊 Fold 4 - Acc: 0.524, F1: 0.000, AUC: 0.491
📊 Fold 5 - Acc: 0.476, F1: 0.645, AUC: 0.518

📋 Results for GAT - MULTINEXT:
Model   Network    Fold  Accuracy    F1  Precision  Recall   AUC
  GAT MULTINEXT       1     0.476 0.000      0.000     0.0 0.436
  GAT MULTINEXT       2     0.476 0.000      0.000     0.0 0.391
  GAT MULTINEXT       3     0.476 0.645      0.476     1.0 0.355
  GAT MULTINEXT       4     0.524 0.000      0.000     0.0 0.491
  GAT MULTINEXT       5     0.476 0.645      0.476     1.0 0.518
  GAT MULTINEXT Average     0.486 0.258      0.190     0.4 0.438
✅ Appended results to F:/Cursos/UOC Master Bio Inf. Est/M0.209 - UOC - TFM - Bio ML - Grafos/DATOS/MULTILAYER/DADES_NAPLES/graph_metrics/nodal_summary_mlayer/figures/ML\MULTINEXT_GNN_metrics_folds.csv
